<a href="https://colab.research.google.com/github/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model/blob/main/notebooks/colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ACM Research 2026 - Colab Setup

Track 1: personalize Maia-2 on a single player's games.

**Run order:** cells 1-4 every new session (Colab wipes everything on disconnect).
Cells 5+ only when you need them.

Runtime: **CPU** for data prep and evaluation. Switch to **T4 GPU** only for fine-tuning.

## 1. Environment check

In [ ]:
!nvidia-smi
import sys
print(sys.version)

/bin/bash: line 1: nvidia-smi: command not found
3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


## 2. Mount Drive (persistent storage)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/acm_chess'
!mkdir -p {PROJ}/data {PROJ}/checkpoints {PROJ}/runs
print(PROJ)

Mounted at /content/drive
/content/drive/MyDrive/acm_chess


## 3. Install


In [ ]:
%cd /content
![ -d repo ] || git clone https://github.com/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model.git repo
%cd /content/repo
!git pull -q
!pip install -q --ignore-requires-python -r requirements.txt

D = f'{PROJ}/data'
!mkdir -p {D}/raw {D}/moves {D}/splits
!ln -sfn {D}/raw data/raw
!ln -sfn {D}/moves data/moves
!ln -sfn {D}/splits data/splits
!ln -sfn {PROJ}/runs data/runs

/content
Cloning into 'repo'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 130 (delta 42), reused 84 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 260.47 KiB | 14.47 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/repo
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
torchvision 0.26.0+cu130 requires torch==2.11.0, but you have torch 2.8.0 which is incompatible.


In [ ]:
!python track1_finetune/F3.py

Loading track1_finetune/exampleGame.csv
Starting Baseline Evaluation on CPU
Downloading...
From (original): https://drive.google.com/uc?id=1gbC1-c7c0EQOPPAVpGWubezeEW8grVwc
From (redirected): https://drive.google.com/uc?id=1gbC1-c7c0EQOPPAVpGWubezeEW8grVwc&confirm=t&uuid=17e01b77-a1ea-4f8a-a731-17dde0389c79
To: /content/repo/maia2_models/.rapid_model.pt.hp2ypo1n.part
100% 280M/280M [00:02<00:00, 119MB/s]
Model for rapid games loaded to cpu.
Processing 63 positions
100% 1/1 [00:01<00:00,  1.67s/it]

CPU | 63 pos | 37.1 pos/sec | Top-1: 34.9% | Top-3: 60.3%

Evaluation: 63 positions  (CI resamples moves (no game_id column))
  top-1         34.9%   95% CI  23.8 -  46.1   n=63
  top-3         60.3%   95% CI  47.6 -  73.0   n=63
  (no 'ply' column, so no per-phase breakdown)


## 4. Get the data. DONT RERUN THIS

Lichess handed over data as PGN, over the API.


In [ ]:
# DONT RERUN THIS. Only needed once: data is already in Drive after the first run.
# Test on one player first, then run everyone. Already-downloaded players are skipped.
!echo thakt > /tmp/one.txt
!python shared/p5_prepare_all_players.py /tmp/one.txt
# !python shared/p5_prepare_all_players.py config/eval_players.txt


[1/1] thakt
  $ /content/repo/shared/download_lichess_games.py thakt
saved: thakt_rapid.pgn
games: 4272
oldest: 2020-11-17
newest: 2026-11-01
  $ /content/repo/shared/build_dataset.py data/raw/thakt_rapid.pgn --target thakt --only-target --min-ply 11 --min-clock 30 -o data/moves/thakt_moves.csv
read            4272 games (0 skipped: no result or no Elo)
dropped         123766 moves by the opponent
dropped         21294 moves with ply < 11
dropped         619 moves with clock < 30.0s
wrote           101949 rows -> data/moves/thakt_moves.csv
date range      2020.11.17 .. 2026.11.01
  $ /content/repo/shared/p7_split_dataset.py data/moves/thakt_moves.csv --out-dir data/splits/thakt
data/moves/thakt_moves.csv: 4230 games, 101949 rows -> data/splits/thakt
  test                   846 games   20733 rows
  val                    338 games    8120 rows
  train_all             3046 games   73096 rows
  train_20_seed0          20 games     527 rows
  train_50_seed0          50 games    1187 rows

In [ ]:
# Only needed once: data is already in Drive after the first run.
# Already-downloaded players are skipped.
# !echo thakt > /tmp/one.txt
# !python shared/p5_prepare_all_players.py /tmp/one.txt
!python shared/p5_prepare_all_players.py config/eval_players.txt


[1/10] thakt
  already downloaded: data/raw/thakt_rapid.pgn
  $ /content/repo/shared/build_dataset.py data/raw/thakt_rapid.pgn --target thakt --only-target --min-ply 11 --min-clock 30 -o data/moves/thakt_moves.csv
read            4272 games (0 skipped: no result or no Elo)
dropped         123766 moves by the opponent
dropped         21294 moves with ply < 11
dropped         619 moves with clock < 30.0s
wrote           101949 rows -> data/moves/thakt_moves.csv
date range      2020.11.17 .. 2026.11.01
  $ /content/repo/shared/p7_split_dataset.py data/moves/thakt_moves.csv --out-dir data/splits/thakt
data/moves/thakt_moves.csv: 4230 games, 101949 rows -> data/splits/thakt
  test                   846 games   20733 rows
  val                    338 games    8120 rows
  train_all             3046 games   73096 rows
  train_20_seed0          20 games     527 rows
  train_50_seed0          50 games    1187 rows
  train_200_seed0        200 games    4795 rows
  train_1000_seed0      1000 game

In [ ]:
## 5. Baseline: base Maia-2 on every player's test set
import glob, os
!mkdir -p {PROJ}/baselines
for d in sorted(glob.glob('data/splits/*')):
    p = os.path.basename(d)
    print(f'\n===== {p} =====')
    !python track1_finetune/F3.py {d}/test.csv -o {PROJ}/baselines/{p}.csv --device auto


===== AlphGC =====
Loading data/splits/AlphGC/test.csv
Starting Baseline Evaluation on AUTO
Downloading...
From (original): https://drive.google.com/uc?id=1gbC1-c7c0EQOPPAVpGWubezeEW8grVwc
From (redirected): https://drive.google.com/uc?id=1gbC1-c7c0EQOPPAVpGWubezeEW8grVwc&confirm=t&uuid=00744c29-04b5-4d14-8260-5b16c3701135
To: /content/repo/maia2_models/.rapid_model.pt.qcint_z4.part
100% 280M/280M [00:05<00:00, 53.7MB/s]
Model for rapid games loaded to cuda.
Processing 18205 positions
100% 18/18 [00:17<00:00,  1.05it/s]

AUTO | 18205 pos | 996.8 pos/sec | Top-1: 55.1% | Top-3: 82.4%

Evaluation: 18205 positions, 730 games  (CI resamples games)
  top-1         55.1%   95% CI  54.4 -  55.8   n=18205
  top-3         82.4%   95% CI  81.8 -  83.0   n=18205
  top-1 by phase:
    opening     57.1%   95% CI  55.4 -  58.7   n=3578
    middlegame  52.7%   95% CI  51.7 -  53.7   n=10301
    endgame     59.3%   95% CI  57.6 -  61.0   n=4326

===== DanhLaThuaThuaThua =====
Loading data/splits/Danh

## 6. Load Maia-2 and confirm it runs

In [ ]:
from maia2 import dataset, inference, model

maia2_model = model.from_pretrained(type="rapid", device="auto")
data = dataset.load_example_test_dataset()

data, accuracy = inference.inference_batch(
    data,
    maia2_model,
    verbose=1,
    batch_size=1024,
    num_workers=0,
)
print('baseline move-matching accuracy on example set:', accuracy)

## 8. Checkpoint pattern

Colab disconnects. Save every epoch to Drive.

In [ ]:
import torch, json, time

def save_ckpt(model, optimizer, epoch, run_name, metrics=None):
    path = f'{PROJ}/checkpoints/{run_name}_epoch{epoch:02d}.pt'
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'opt': optimizer.state_dict(),
        'metrics': metrics or {},
        'ts': time.time(),
    }, path)
    with open(f'{PROJ}/runs/{run_name}.jsonl', 'a') as f:
        f.write(json.dumps({'epoch': epoch, **(metrics or {})}) + '\n')
    return path

## 9. Freeze the environment for the writeup

Run this at the end of any run whose numbers go in the report.

In [ ]:
!pip freeze > {PROJ}/runs/requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv > {PROJ}/runs/gpu.txt
print('frozen')